# New Prefix Analysis — Deaggregation vs. New Space

Classifies the new prefixes that triggered each critical exceedance event
(those losing at least one Tier-1 or Major peer) as either:

- **Pure deaggregation**: every new prefix is a subnet of a block already in the pre-event set
- **Pure new space**: every new prefix falls outside any previously announced block
- **Mixed**: the event contains both types — further split into:
  - *deagg-dominant* (≥80% of new prefixes are deaggregation)
  - *truly mixed* (20%–80%)
  - *new-space-dominant* (≤20% are deaggregation)

## Imports

In [ ]:
import os
import json
import datetime
import ipaddress
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


## Configuration

In [ ]:
font_size = 24
scale_factor = 1.2
two_sided_font_size = font_size * scale_factor

plt.rcParams["font.size"] = font_size
ipvs = [4, 6]
ipv_color = {4: "tab:blue", 6: "tab:green"}


In [ ]:
REPO_ROOT = os.path.abspath("..")
with open(os.path.join(REPO_ROOT, "settings.json")) as f:
    parameters = json.load(f)
    for _k in ("DATA_DIR", "DATA_RAW_DIR", "IMAGE_DIR", "WORKING_DIR", "VISIBILITY_OUTPUT_DIR", "VISIBILITY_ANNOUNCED_OUTPUT_DIR"):
        if isinstance(parameters.get(_k), str) and not os.path.isabs(parameters[_k]):
            parameters[_k] = os.path.normpath(os.path.join(REPO_ROOT, parameters[_k]))

data_dir  = parameters["DATA_DIR"]
image_dir = parameters["IMAGE_DIR"]


In [ ]:
# Thresholds for reclassifying mixed events
# Events where >= DEAGG_DOMINANT_PCT% of new prefixes are deagg → deagg-dominant
# Events where <= NEW_SPACE_DOMINANT_PCT% of new prefixes are deagg → new-space-dominant
DEAGG_DOMINANT_PCT    = 80
NEW_SPACE_DOMINANT_PCT = 20


In [ ]:
# Category colours: deagg side → greens, mixed → gray, new-space side → oranges/reds
# Ordered left-to-right: most-deagg → most-new-space
LABEL_ORDER = [
    "pure_deaggregation",
    "deagg_dominant",
    "truly_mixed",
    "new_space_dominant",
    "pure_new_space",
]


# Output directory for this notebook's figures
out_dir = f"{image_dir}/new_prefix"
os.makedirs(out_dir, exist_ok=True)
print(f"Figure output: {out_dir}")


## Open Stats Output File

In [ ]:
## Open stats output file (overwrites on every run)
numbers_dir = f"{data_dir}/processed/numbers"
os.makedirs(numbers_dir, exist_ok=True)

_stats = open(f"{numbers_dir}/13-new_prefix.md", "w")
_stats.write("# Stats: 13-new_prefix\n\n")
_stats.write(f"*Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}*\n\n")
_stats.write(f"- Deagg-dominant threshold   : >= {DEAGG_DOMINANT_PCT}%\n")
_stats.write(f"- New-space-dominant threshold: <= {NEW_SPACE_DOMINANT_PCT}%\n\n")

print("Stats file opened.")


## Load Data

We only need the critical exceedance events JSON — each entry already contains
`previous_prefixes` (the prefix set one snapshot before crossing) and
`new_prefixes` (prefixes that appeared at the crossing snapshot).

In [ ]:
critical_events_filename = f"{data_dir}/processed/critical_excedence_events.json"
with open(critical_events_filename) as f:
    data = json.load(f)

#convert keys to int
data = {int(k): v for k, v in data.items()}

for ip_version in ipvs:
    n = len(data[ip_version])
    has_prefixes = sum(1 for e in data[ip_version] if e.get("new_prefixes"))
    print(f"IPv{ip_version}: {n} critical events, {has_prefixes} with new_prefixes populated")


## Classification Functions

`analyze_event` does the subnet check for every new prefix against the previous prefix set.

`classify_event` maps the numeric deagg fraction to a label using the configured thresholds.

In [ ]:
def analyze_event(event):
    """Return per-event deagg / new-space breakdown.

    For each prefix in new_prefixes, check whether it is a subnet of any
    prefix in previous_prefixes using ipaddress.ip_network.subnet_of().

    Returns a dict or None if there are no parseable new prefixes.
    """
    prev = event.get("previous_prefixes", [])
    new  = event.get("new_prefixes", [])
    if not new:
        return None

    # Parse previous prefixes into ip_network objects for subnet_of() comparisons
    prev_nets = []
    for p in prev:
        try:
            prev_nets.append(ipaddress.ip_network(p, strict=False))
        except ValueError:
            pass  # skip any malformed strings

    deagg_prefixes, new_space_prefixes = [], []
    for p in new:
        try:
            net = ipaddress.ip_network(p, strict=False)
        except ValueError:
            continue
        # subnet_of() returns True if net is contained within pn (deaggregation)
        if any(net.subnet_of(pn) for pn in prev_nets):
            deagg_prefixes.append(net)
        else:
            new_space_prefixes.append(net)

    total = len(deagg_prefixes) + len(new_space_prefixes)
    if total == 0:
        return None

    return {
        "deagg"    : deagg_prefixes,
        "new_space": new_space_prefixes,
        "deagg_pct": len(deagg_prefixes) / total * 100,
        "total"    : total,
    }


In [ ]:
def classify_event(result):
    """Map an analyze_event result to a coarse label."""
    if result is None:
        return "no_data"
    if not result["new_space"]:
        # every new prefix is a subnet of a previous one
        return "pure_deaggregation"
    if not result["deagg"]:
        # no new prefix overlaps with previous space
        return "pure_new_space"
    # mixed: apply the dominance thresholds
    if result["deagg_pct"] >= DEAGG_DOMINANT_PCT:
        return "deagg_dominant"
    if result["deagg_pct"] <= NEW_SPACE_DOMINANT_PCT:
        return "new_space_dominant"
    return "truly_mixed"


## Run Analysis

Collect labels and prefix-length distributions for both IP versions.

In [ ]:
results = {}

for ip_version in ipvs:
    events = data[ip_version]

    analyses = [analyze_event(e) for e in events]
    labels   = [classify_event(r) for r in analyses]

    # Accumulate prefix lengths separately for deagg and new-space prefixes
    deagg_prefixlens     = []
    new_space_prefixlens = []

    # Collect detail for mixed events (those with both deagg and new-space components)
    mixed_detail = []  # list of (asn, deagg_pct, total_new, label)

    for e, r, label in zip(events, analyses, labels):
        if r is None:
            continue
        for net in r["deagg"]:
            deagg_prefixlens.append(net.prefixlen)
        for net in r["new_space"]:
            new_space_prefixlens.append(net.prefixlen)
        if label in ("deagg_dominant", "new_space_dominant", "truly_mixed"):
            mixed_detail.append(
                (e["excedence_event"]["asn"], r["deagg_pct"], r["total"], label)
            )

    results[ip_version] = {
        "total_events"        : len(events),
        "label_counts"        : Counter(labels),
        "deagg_prefixlens"    : deagg_prefixlens,
        "new_space_prefixlens": new_space_prefixlens,
        "mixed_detail"        : mixed_detail,
    }

print("Analysis complete.")


## Results — Event Classification Summary

In [ ]:
for ip_version in ipvs:
    r = results[ip_version]
    total  = r["total_events"]
    counts = r["label_counts"]

    print(f"\n=== IPv{ip_version} ({total} critical events) ===")
    for label in LABEL_ORDER:
        n = counts.get(label, 0)
        print(f"  {label:<22}: {n:3d}  ({100*n/total:5.1f}%)")

    # Collapsed view: group pure + dominant on each side
    deagg_side = counts.get("pure_deaggregation", 0) + counts.get("deagg_dominant", 0)
    new_side   = counts.get("pure_new_space", 0)     + counts.get("new_space_dominant", 0)
    mixed      = counts.get("truly_mixed", 0)
    print(f"  --- collapsed ---")
    print(f"  Deagg-side (pure + dominant) : {deagg_side:3d}  ({100*deagg_side/total:.1f}%)")
    print(f"  New-space-side (pure + dom.) : {new_side:3d}  ({100*new_side/total:.1f}%)")
    print(f"  Truly mixed                  : {mixed:3d}  ({100*mixed/total:.1f}%)")


## Results — Prefix Length Distributions

In [ ]:
for ip_version in ipvs:
    r = results[ip_version]
    print(f"\n=== IPv{ip_version} ===")

    for kind, prefixlens in [
        ("Deaggregation", r["deagg_prefixlens"]),
        ("New space",     r["new_space_prefixlens"]),
    ]:
        if not prefixlens:
            print(f"  {kind}: no data")
            continue
        c = Counter(prefixlens)
        pl_total = len(prefixlens)
        print(f"  {kind} ({pl_total} prefixes total):")
        for pl, cnt in sorted(c.items()):
            bar = "#" * int(cnt / pl_total * 40)  # quick ASCII bar
            print(f"    /{pl:>3}: {cnt:4d}  ({100*cnt/pl_total:5.1f}%)  {bar}")


## Results — Mixed Event Detail

In [ ]:
for ip_version in ipvs:
    r = results[ip_version]
    print(f"\n=== IPv{ip_version} mixed events ===")
    if not r["mixed_detail"]:
        print("  None")
        continue
    # Sort by deagg_pct descending so deagg-dominant events appear first
    for asn, deagg_pct, total_new, label in sorted(
        r["mixed_detail"], key=lambda x: -x[1]
    ):
        print(f"  ASN {asn:>8}: {deagg_pct:5.0f}% deagg of {total_new:3d} new prefixes  [{label}]")


## Plots

In [ ]:
LABEL_COLORS = {
    "pure_deaggregation" : "#2ca02c",   # dark green
    "deagg_dominant"     : "#98df8a",   # light green
    "truly_mixed"        : "#adb5bd",   # neutral gray
    "new_space_dominant" : "#ffbb78",   # light orange
    "pure_new_space"     : "#d62728",   # dark red-orange
}
LABEL_DISPLAY = {
    "pure_deaggregation" : "Deaggregation",
    "deagg_dominant"     : "Deaggregation-dominant",
    "truly_mixed"        : "Mixed",
    "new_space_dominant" : "New-space-dominant",
    "pure_new_space"     : "New space",
}


In [ ]:
## Figure 1 — Stacked horizontal bar: event classification

plt.figure(figsize=(12, 5))

for ipv_index, ipv in enumerate(ipvs):
    total  = results[ipv]["total_events"]
    counts = results[ipv]["label_counts"]

    left = 0  # running left edge for each stacked segment
    for label in LABEL_ORDER:
        n = counts[label]
        if n == 0:
            continue

        width = n / total * 100  # convert to percentage width

        plt.barh(
            ipv_index, width, left=left, 
            height=.5,
            align="center",
            alpha=0.8,
            color=LABEL_COLORS[label],
            label=LABEL_DISPLAY[label] if ipv_index == 0 else None,  # show label only for first bar
        )
        # Annotate count inside the bar if wide enough to fit
        if width > 5:
            plt.text(
                left + width / 2, ipv_index,
                f"{width:.1f}%",
                ha="center", va="center",
                fontsize=.9*font_size, 
                color="black", 
                # fontweight="bold",
            )
        left += width

# Y-axis labels
plt.yticks(range(len(ipvs)), [f"IPv{v}" for v in ipvs], fontsize=font_size)

plt.xlabel("Share of critical events (%)", fontsize=font_size)
plt.xlim(0, 100)
plt.grid(axis="x", alpha=0.3)
# plt.spines[["top", "right", "left"]].set_visible(False)

plt.legend(
    loc="lower right",
    bbox_to_anchor=(1, 1),
    ncol=2,
    fontsize=font_size,
    frameon=False,
)

plt.tight_layout()
plt.savefig(f"{image_dir}/new_prefixes/event_classification_stacked.pdf", bbox_inches="tight", dpi=300)
plt.savefig(f"{image_dir}/new_prefixes/event_classification_stacked.png", bbox_inches="tight", dpi=300)
plt.show()


In [ ]:
# fig, axes = plt.subplots(2, 1, figsize=(10, 10))

# panel_config = [
#     # (axis, ip_version, kind_key, color, title)
#     (axes[0], "4", "new_space_prefixlens", "#d62728", "IPv4 — New space prefix lengths"),
#     (axes[1], "6", "deagg_prefixlens",     "#2ca02c", "IPv6 — Deaggregation prefix lengths"),
# ]

# for ax, ip_version, kind_key, color, title in panel_config:
#     prefixlens = results[ip_version][kind_key]
#     c = Counter(prefixlens)
#     pl_total = len(prefixlens)

#     lengths = sorted(c.keys())
#     # Normalise to percentage of prefixes in this category
#     pcts    = [c[pl] / pl_total * 100 for pl in lengths]
#     x_ticks = np.arange(len(lengths))

#     bars = ax.bar(x_ticks, pcts, color=color, alpha=0.85, edgecolor="white")

#     # Annotate percentage on top of each bar
#     for bar, pct in zip(bars, pcts):
#         if pct > 3:   # only annotate bars tall enough to read
#             ax.text(
#                 bar.get_x() + bar.get_width() / 2,
#                 bar.get_height() + 0.5,
#                 f"{pct:.0f}%",
#                 ha="center", va="bottom", fontsize=11,
#             )

#     ax.set_xticks(x_ticks)
#     ax.set_xticklabels([f"/{pl}" for pl in lengths], fontsize=13)
#     ax.set_ylabel("Share of prefixes (%)", fontsize=font_size)
#     ax.set_title(title, fontsize=font_size, pad=10)
#     ax.grid(axis="y", alpha=0.3)
#     ax.spines[["top", "right"]].set_visible(False)
#     ax.set_ylim(0, max(pcts) * 1.15)  # headroom for annotations

# plt.tight_layout()
# plt.savefig(f"{image_dir}/new_prefixes/prefix_length_distributions.pdf", bbox_inches="tight", dpi=300)
# plt.savefig(f"{image_dir}/new_prefixes/prefix_length_distributions.png", bbox_inches="tight", dpi=300)
# plt.show()


## Write Stats File

In [ ]:
## Event classification table — one section per IP version
for ip_version in [4, 6]:
    r = results[ip_version]
    total  = r["total_events"]
    counts = r["label_counts"]

    _stats.write(f"## IPv{ip_version} — {total} critical events\n\n")

    # Fine-grained classification table
    _stats.write("### Event classification\n\n")
    _stats.write("| Label | N | % |\n")
    _stats.write("|-------|---|---|\n")
    for label in LABEL_ORDER:
        n = counts.get(label, 0)
        _stats.write(f"| {label} | {n} | {100*n/total:.1f}% |\n")

    # Collapsed view
    deagg_side = counts.get("pure_deaggregation", 0) + counts.get("deagg_dominant", 0)
    new_side   = counts.get("pure_new_space", 0)     + counts.get("new_space_dominant", 0)
    mixed      = counts.get("truly_mixed", 0)
    _stats.write("\n### Collapsed\n\n")
    _stats.write(f"- Deagg-side (pure + dominant)   : {deagg_side} ({100*deagg_side/total:.1f}%)\n")
    _stats.write(f"- New-space-side (pure + dominant): {new_side} ({100*new_side/total:.1f}%)\n")
    _stats.write(f"- Truly mixed                    : {mixed} ({100*mixed/total:.1f}%)\n\n")

_stats.flush()
print("Classification written.")


In [ ]:
## Prefix length distributions
for ip_version in ipvs:
    r = results[ip_version]
    _stats.write(f"### IPv{ip_version} — Prefix length distributions\n\n")

    for kind, prefixlens in [
        ("Deaggregation", r["deagg_prefixlens"]),
        ("New space",     r["new_space_prefixlens"]),
    ]:
        if not prefixlens:
            _stats.write(f"**{kind}**: no data\n\n")
            continue
        c = Counter(prefixlens)
        pl_total = len(prefixlens)
        _stats.write(f"**{kind}** ({pl_total} prefixes):\n\n")
        _stats.write("| Prefix length | N | % |\n")
        _stats.write("|--------------|---|---|\n")
        for pl, cnt in sorted(c.items()):
            _stats.write(f"| /{pl} | {cnt} | {100*cnt/pl_total:.1f}% |\n")
        _stats.write("\n")

_stats.flush()
print("Prefix length distributions written.")


In [ ]:
## Mixed event detail per IP version
for ip_version in ipvs:
    r = results[ip_version]
    _stats.write(f"### IPv{ip_version} — Mixed event detail\n\n")

    if r["mixed_detail"]:
        _stats.write("| ASN | Deagg% | New prefixes | Label |\n")
        _stats.write("|-----|--------|--------------|-------|\n")
        # Sort by deagg_pct descending
        for asn, deagg_pct, total_new, label in sorted(
            r["mixed_detail"], key=lambda x: -x[1]
        ):
            _stats.write(f"| {asn} | {deagg_pct:.0f}% | {total_new} | {label} |\n")
    else:
        _stats.write("None\n")
    _stats.write("\n")

_stats.flush()
_stats.close()
print(f"Stats written to {numbers_dir}/13-new_prefix.md")
